# Day 4 — Structured coverage ingestion lab

Ingest synthetic plans/claims CSVs with pandas, load them into SQLite `coverage.db`, and run 5 member-facing SQL queries.

**Deliverables (verified at these paths):** `data/plans.csv`, `data/claims.csv`, `coverage.db`, `structured_queries.md`.

Run this notebook from `notebooks/` so relative paths `../data/` and `../coverage.db` resolve. Copy each query + output into `../structured_queries.md` when done.

In [ ]:
import sqlite3
from pathlib import Path

import pandas as pd

DATA_DIR = Path("../data")
DB_PATH = Path("../coverage.db")

DATA_DIR.mkdir(parents=True, exist_ok=True)

## 1. Load CSVs

Create `../data/plans.csv` and `../data/claims.csv` first (see mission brief for required columns). Include a **Gold PPO** plan and member **M1001**.

In [ ]:
plans = pd.read_csv(DATA_DIR / "plans.csv")
claims = pd.read_csv(DATA_DIR / "claims.csv")

plans.info()
plans.head()

In [ ]:
claims.info()
claims.head()

## 2. Clean

TODO: drop duplicates, fill/drop nulls, coerce `date_filed` to datetime.

In [ ]:
plans = plans.drop_duplicates()
claims = claims.drop_duplicates()

# TODO: handle nulls (fill or drop) as needed for your synthetic data
claims["date_filed"] = pd.to_datetime(claims["date_filed"], errors="coerce")

print(f"plans rows: {len(plans)}, claims rows: {len(claims)}")

## 3. Load into SQLite (`coverage.db`)

In [ ]:
conn = sqlite3.connect(DB_PATH)
plans.to_sql("plans", conn, if_exists="replace", index=False)
claims.to_sql("claims", conn, if_exists="replace", index=False)
print(f"Wrote {DB_PATH.resolve()}")

## 4. Five SQL queries

Fill in / adjust each query, run it, then paste query + output into `../structured_queries.md`.

In [ ]:
# 1. What's the deductible on the Gold PPO plan?
q1 = """
SELECT plan_name, annual_deductible
FROM plans
WHERE plan_name LIKE '%Gold%PPO%'
"""
pd.read_sql_query(q1, conn)

In [ ]:
# 2. How many claims are pending for member M1001?
q2 = """
SELECT COUNT(*) AS pending_count
FROM claims
WHERE member_id = 'M1001' AND LOWER(status) = 'pending'
"""
pd.read_sql_query(q2, conn)

In [ ]:
# 3. Which plans have a monthly premium under $400?
q3 = """
SELECT plan_id, plan_name, monthly_premium
FROM plans
WHERE monthly_premium < 400
"""
pd.read_sql_query(q3, conn)

In [ ]:
# 4. JOIN claims and plans
q4 = """
SELECT c.claim_id, c.member_id, c.procedure, c.claim_amount, p.plan_name
FROM claims c
JOIN plans p ON c.plan_id = p.plan_id
LIMIT 20
"""
pd.read_sql_query(q4, conn)

In [ ]:
# 5. Top-N most claimed procedures
q5 = """
SELECT procedure, COUNT(*) AS claim_count
FROM claims
GROUP BY procedure
ORDER BY claim_count DESC
LIMIT 5
"""
pd.read_sql_query(q5, conn)

In [ ]:
conn.close()
print("Done — copy queries + outputs into ../structured_queries.md, then commit and push.")